# Predikcija popularnosti pesama na Spotify-u

**Autori:** Isidora Pavlović E2 131/2025, Nikola Spasojević E2 82/2025

## 1. Pregled i eksplorativna analiza podataka

Dataset sadrži 114,000 pesama iz 125 žanrova sa sledećim obeležjima:

**Audio karakteristike (13):**
- `danceability` - pogodnost za ples
- `energy` - energičnost
- `loudness` - glasnoća
- `speechiness` - prisustvo govora
- `acousticness` - akustičnost
- `instrumentalness` - instrumentalnost
- `liveness` - prisustvo publike
- `valence` - valentnost (pozitivnost)
- `tempo` - tempo
- `duration_ms` - trajanje u milisekundama
- `key` - tonalitet
- `mode` - modalitet (dur/mol)
- `time_signature` - takt

**Metapodaci:**
- `track_name`, `artists`, `album_name`
- `track_genre` - muzički žanr
- `explicit` - da li pesma ima eksplicitan sadržaj

**Ciljno obeležje:**
- `popularity` - popularnost pesme (0-100), izračunata Spotify algoritmom na osnovu broja reprodukcija i njihove skorašnjosti

In [ ]:
# Učitavanje podataka
df = pd.read_csv('dataset.csv')
df = df.dropna(subset=['artists', 'album_name', 'track_name'])

print(f"Ukupno pesama: {len(df):,}")
print(f"Broj žanrova: {df['track_genre'].nunique()}")

## 2. Ključna otkrića iz analize

### 2.1 Distribucija popularnosti

Analiza je pokazala da:
- **Većina pesama ima nisku popularnost** - distribucija je nagnuta levo
- Srednja vrednost: ~33, medijana: ~28
- Veliki deo pesama (>40%) ima popularnost ispod 25

### 2.2 Korelacija sa audio karakteristikama

Najjače korelacije sa popularnošću:
- **Pozitivne:** `loudness` (+0.27), audio karakteristike visokog energetskog nivoa
- **Negativne:** `acousticness` (-0.21), `instrumentalness` (-0.18)

**Važno:** Sve korelacije su slabe (|r| < 0.3), što ukazuje da popularnost zavisi od mnogo drugih faktora

### 2.3 Uticaj žanra

ANOVA test: **Žanr značajno utiče na popularnost** (p < 0.001)
- Različiti žanrovi imaju različite prosečne popularnosti
- Jazz, klasična muzika - niža popularnost
- Pop, dance - viša popularnost

In [ ]:
![Distribucija popularnosti](viz_1_distribucija_popularnosti.png)

**Napomena:** Sve vizualizacije u ovom notebook-u su generisane pomoću `generate_visualizations.py` skripte koja obezbeđuje:
- Konzistentnu stilizaciju svih grafika
- Reproduktivnost rezultata
- PNG slike visokog kvaliteta (300 DPI) pogodne za prezentacije i izveštaje

### 2.4 Analiza šablona i klastera

Pored analize pojedinačnih atributa, važno je identifikovati **dublje šablone** i **klastere** u podacima. Primenjujemo K-means klasterizaciju da otkrijemo grupe pesama sa sličnim karakteristikama i analiziramo kako se klasteri razlikuju po popularnosti.

**Metodologija:**

1. **K-means algoritam** - deli podatke u grupe na osnovu sličnosti audio karakteristika
2. **StandardScaler normalizacija** - sve karakteristike dobijaju jednaku težinu (mean=0, std=1)
3. **PCA (Principal Component Analysis)** - smanjuje dimenzionalnost sa 7D na 2D za vizualizaciju
4. **Analiza profila** - svaki klaster ima specifičan "profil" audio karakteristika
5. **Analiza interakcija** - ispitivanje kako features zajedno utiču na popularnost

Umesto fokusa na pojedinačne korelacije (npr. "loudness +0.27"), otkrivamo **kombinacije karakteristika** koje definišu uspešne šablone.

In [ ]:
# K-means klasterizacija
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Normalizacija audio features
audio_cols = ['danceability', 'energy', 'loudness', 'acousticness', 
              'instrumentalness', 'valence', 'tempo']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[audio_cols])

# K-means sa 4 klastera
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(df_scaled)

# Analiza klastera
cluster_stats = df.groupby('cluster').agg({
    'popularity': ['mean', 'median', 'count'],
    'danceability': 'mean',
    'energy': 'mean',
    'loudness': 'mean',
    'acousticness': 'mean'
}).round(2)

print("Karakteristike klastera:")
print(cluster_stats)
print(f"\nPronadjena su {df['cluster'].nunique()} klastera pesama")

In [ ]:
![Vizualizacija klastera](viz_2_klasteri.png)

**Napomena o PCA:**
- Smanjuje dimenzionalnost sa 7 audio features na 2 dimenzije za vizualizaciju
- Prve 2 glavne komponente obuhvataju najveći deo varijabilnosti podataka
- Omogućava 2D prikaz višedimenzionalnih klastera

**Otkriveni šabloni - Prirodni klasteri pesama:**

K-means algoritam identifikovao je **4 distinktne grupe pesama** na osnovu kombinacije audio karakteristika:

### **Klaster 0 - Metal/Energični (30.2% pesama, popularnost: 34.1)**
**Ključne kombinacije:**
- ↑↑ **Visoka energy (+27%)** + **instrumentalness (+40%)**
- ↓↓ Niska acousticness (-79%), valence (-33%), loudness (-28%)
- ↑ Iznad proseka: tempo (+12%)

**Profil:** Hard/metal žanrovi (grindcore, death-metal, black-metal, metalcore)  
**Karakteristike:** Energične, instrumentalne pesme sa teškim zvukom ali tihijim glasnoćom od očekivanja

---

### **Klaster 1 - Akustični/Tihi (23.4% pesama, popularnost: 34.2 - NAJVIŠA)**
**Ključne kombinacije:**
- ↑↑ **Visoka acousticness (+113%)** + **loudness (+28%)**
- ↓↓ Niska energy (-38%), instrumentalness (-72%)
- ↓ Ispod proseka: valence (-15%), danceability (-5%)

**Profil:** Akustični žanrovi (tango, romance, honky-tonk, jazz, comedy)  
**Karakteristike:** Tihe, akustične pesme sa vokalima  
**Interesantno:** Uprkos niskoj energiji, ovaj klaster ima NAJVEĆU prosečnu popularnost! To pokazuje da akustična kombinacija ima svoj jaki market.

---

### **Klaster 2 - Ambijent/Instrumentalni (7.5% pesama, popularnost: 28.6 - NAJNIŽA)**
**Ključne kombinacije:**
- ↑↑ **Ekstremno visoka instrumentalness (+405%)** + **acousticness (+158%)** + **loudness (+146%)**
- ↓↓ Sve ostalo nisko: energy (-68%), danceability (-35%), valence (-60%)

**Profil:** Ambijentalni žanrovi (sleep, new-age, classical, ambient, piano)  
**Karakteristike:** Veoma tihe, instrumentalne, mirne kompozicije  
**Interesantno:** Najmanji klaster sa najnižom popularnošću - niša tržište

---

### **Klaster 3 - Plesni/Veseli (38.9% pesama, popularnost: 32.9)**
**Ključne kombinacije:**
- ↑↑ **Visoka danceability (+22%)** + **valence (+46%)**
- ↓↓ Niska acousticness (-37%), instrumentalness (-66%), loudness (-23%)
- ≈ Energy iznad proseka (+15%), tempo prosečan

**Profil:** Plesni žanrovi (reggaeton, reggae, latino, dancehall, forró)  
**Karakteristike:** Vesele, plesne pesme sa vokalima  
**Interesantno:** Najveći klaster (39% svih pesama) - mainstream popularna muzika

---

### **Ključni zaključci o kombinacijama:**

1. **Akustične + glasne** pesme (Klaster 1) postižu NAJVIŠU prosečnu popularnost uprkos niskoj energiji
2. **Danceability + valence** (Klaster 3) kombinacija dominira tržištem - 39% svih pesama
3. **Instrumentalne** pesme (Klaster 2) imaju najnižu popularnost - niša
4. **Energy ≠ glasnoća**: Klaster 0 ima visoku energy ali nižu glasnoću (metal žanrovi)

**Zaključak:** Popularnost zavisi od **šablona** (specifične kombinacije karakteristika), ne pojedinačnih atributa. Različiti šabloni mogu biti uspešni - akustični tihi zvuk (Klaster 1) i energični plesni zvuk (Klaster 3) oba dostižu visoku popularnost, ali kroz potpuno različite kombinacije karakteristika!



In [ ]:
![Interakcija Energy-Loudness po klasterima](viz_3_interakcije_klastera.png)

**O grafikonu:**
- Svaki subplot prikazuje jedan klaster
- X osa: Energy, Y osa: Loudness
- Boja tačke: Popularnost (zeleno = visoka, žuto = srednja, crveno = niska)
- Pokazuje kako **kombinacija** dva atributa utiče na popularnost **unutar** svakog klastera
- Različiti klasteri imaju različite obrasce i raspodele popularnosti

## 3. Metodologija

Na osnovu analize literature i karakteristika problema, odabrali smo **regresioni pristup** za predikciju tačne vrednosti popularnosti.

### Odabrani algoritmi

**Random Forest Regressor**
- Ensemble metoda bazirana na više stabala odlučivanja
- Robusna na outliers i ne zahteva feature scaling
- Otporna na overfitting
- Efikasna pri radu sa većim brojem ulaznih karakteristika
- Parametri: 100 stabala, maksimalna dubina 20

**XGBoost Regressor**
- Gradient boosting algoritam
- Sekvencijalno gradi model gde svaki novi model ispravlja greške prethodnog
- Brz i efikasan za velike datasete
- Parametri: 100 stabala, learning rate 0.1, maksimalna dubina 6

### Metrike evaluacije

- **Mean Absolute Error (MAE)** - prosečna apsolutna greška
- **R² Score** - koeficijent determinacije, pokazuje koliko model objašnjava varijansu podataka

**Podela podataka:** 80% training, 20% test

In [ ]:
# Priprema podataka
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Audio features + enkodovanje kategoričkih varijabli
audio_features = ['danceability', 'energy', 'loudness', 'acousticness', 
                  'instrumentalness', 'valence', 'tempo', 'duration_ms']
df['genre_encoded'] = LabelEncoder().fit_transform(df['track_genre'])

X = df[audio_features + ['genre_encoded']]
y = df['popularity']

# Podela podataka
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training: {len(X_train):,} | Test: {len(X_test):,} pesama")

In [ ]:
# Treniranje Random Forest i XGBoost modela
rf_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42)
rf_model.fit(X_train, y_train)

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

print("Modeli uspešno trenirani")

## 4. Rezultati i performanse

In [ ]:
# Evaluacija performansi
from sklearn.metrics import mean_absolute_error, r2_score

rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

print("Random Forest:")
print(f"  MAE: {mean_absolute_error(y_test, rf_pred):.3f}")
print(f"  R²:  {r2_score(y_test, rf_pred):.3f}")

print("\nXGBoost:")
print(f"  MAE: {mean_absolute_error(y_test, xgb_pred):.3f}")
print(f"  R²:  {r2_score(y_test, xgb_pred):.3f}")

: 

In [ ]:
![Važnost atributa](viz_4_feature_importance.png)

In [ ]:
![Greška predikcije po klasterima](viz_5_mae_po_klasterima.png)

## 4.5. Analiza kvaliteta podataka: Rešavanje problema sa nulama

### Problem

Tokom analize dataset-a, otkriveno je da **16,019 pesama (14.1%)** ima vrednost popularnosti označenu kao 0. Ovo postavlja pitanje: da li su te pesme zaista potpuno nepopularne, ili se radi o nedostajućim vrednostima?

### Poređenje audio karakteristika

Analiza pokazuje da pesme sa popularnost=0 imaju slične audio karakteristike kao i pesme sa pop>0:

| Karakteristika | Pop=0 Mean | Pop>0 Mean | Razlika |
|---------------|-----------|-----------|---------|
| danceability | 0.574 | 0.566 | +0.008 |
| energy | 0.615 | 0.646 | -0.031 |
| acousticness | 0.341 | 0.311 | +0.030 |
| valence | 0.507 | 0.469 | +0.038 |

**Zaključak**: Pesme sa pop=0 nisu bitno drugačije u audio karakteristikama, što sugeriše da su verovatno nedostajuće vrednosti, a ne potpuno nepopularne pesme.

### Eksperimenti: Dva pristupa

**Eksperiment 1: Uklanjanje pesama sa pop=0**
- Dataset veličina: 97,980 pesama (gubitak 14.1% podataka)
- Treniranje Random Forest modela
- **Rezultati**: MAE = 10.114, R² = 0.477

**Eksperiment 2: KNN Imputation**
- Pristup: Pronalaženje 10 najsličnijih pesama (na osnovu audio karakteristika) i dodeljivanje prosečne popularnosti
- Dataset veličina: 113,999 pesama (zadržani svi podaci)
- Prosečna imputirana vrednost: 37.27
- **Rezultati**: MAE = 9.747, R² = 0.467

In [ ]:
![Poređenje pristupa za rešavanje problem nula](zero_popularity_comparison.png)

### Zaključci analize

**Ključni nalazi:**

1. **KNN Imputation je bolji pristup** - niži MAE uprkos blago nižem R²
   - Poboljšanje MAE: **3.63%** (10.114 → 9.747)
   - Zadržavanje svih 113,999 podataka (bez gubitka 14.1% dataset-a)

2. **Priroda nula**: Analiza audio karakteristika pokazuje da pesme sa pop=0 nisu bitno drugačije od ostalih, što sugeriše da je reč o nedostajućim vrednostima, a ne o zaista nepopularnim pesmama

3. **Distribucija imputiranih vrednosti**: Imputirane vrednosti (mean=37.27, median=38.20) su realne i odgovaraju distribuciji popularnosti u dataset-u

**Preporuka**: Koristiti KNN Imputation pristup za finalni model jer:
- Daje bolje performanse (niži MAE)
- Zadržava sve podatke
- Koristi sličnost audio karakteristika za pametno popunjavanje nedostajućih vrednosti

**Dataset-i sačuvani za dalja istraživanja:**
- `dataset_removed_zeros.csv` (bez pop=0, 97,980 pesama)
- `dataset_knn_imputed.csv` (sa KNN imputation, 113,999 pesama)

## 5. Zaključci i diskusija

### Performanse modela

Finalni modeli (sa KNN Imputation pristupom) pokazuju **solidne performanse**:
- **MAE ~ 9.75**: U proseku, predikcija greši za ~9.75 poena (poboljšanje od 3.63% u odnosu na uklanjanje nula)
- **R² ~ 0.467**: Modeli objašnjavaju ~47% varijanse u popularnosti
- Random Forest i XGBoost postižu slične rezultate

### Kvalitet podataka

**Ključno otkriće**: 14.1% dataset-a (16,019 pesama) ima popularnost=0
- Analiza pokazuje da su verovatno **nedostajuće vrednosti**, ne potpuno nepopularne pesme
- **KNN Imputation pristup** pokazuje bolje rezultate od jednostavnog uklanjanja
- Korišćenje sličnosti audio karakteristika omogućava pametno popunjavanje podataka

### Najvažniji faktori

Analiza važnosti atributa pokazuje da najveći uticaj imaju:
1. **Loudness** (glasnoća) - najjača pozitivna korelacija
2. **Genre** (žanr) - različiti žanrovi imaju različite nivoe popularnosti
3. **Energy, Danceability** - energetske karakteristike pesme
4. **Duration_ms** - trajanje pesme

### Dubinski šabloni i klasteri

Klaster analiza je otkrila važne šablone koji prevazilaze jednostavne korelacije:

**Pronađeni klasteri:**
- Pesame se prirodno grupišu u 4 distinktna klastera sa različitim audio profilima
- **Energične pesme** (visoka energy, loudness) imaju najveću prosečnu popularnost
- **Akustične pesme** formiraju poseban klaster sa umerenom popularnošću
- **Instrumentalne pesme** pokazuju najnižu popularnost i najjasniju separaciju

**Važno:**
- Popularnost zavisi od **kombinacije karakteristika**, ne pojedinačnih atributa
- Postoje **različiti šabloni uspešnosti** - nema jedinstvenog "recepta"
- Interakcije između atributa (npr. energy + loudness) imaju snažniji uticaj od bilo kojeg pojedinačnog atributa

### Ključna saznanja

1. **Kvalitet podataka je kritičan**: Pametno rešavanje nedostajućih vrednosti (KNN Imputation) poboljšava performanse za 3.63%

2. Audio karakteristike **objašnjavaju skoro 50% varijanse** u popularnosti - značajno bolji rezultat od očekivanog (15-18% iz literature)

3. **Glasnije pesme** su u proseku popularnije

4. **Žanr ima signifikantan uticaj** (ANOVA test, p < 0.001)

5. Većina pesama ima **nisku popularnost** (<25), distribucija je neujednačena

6. **Šabloni i interakcije** između atributa su važniji od pojedinačnih korelacija

**Ograničenja:**
- Popularnost zavisi od brojnih faktora koje dataset ne sadrži:
  - Marketing i promocija
  - Brend izvođača i njihova prethodna popularnost
  - Trenutni trendovi i sezonalnost
  - Viralni momenti na društvenim mrežama
  - Uključenost u popularne plejliste

**Finalni zaključak**: Random Forest uz KNN Imputation pristup je pouzdan algoritam za ovaj problem i postiže rezultate značajno bolje od očekivanih iz literature.

